# 10 — Calibração do SVM binário (custo, normalizador e paralelismo)

**Pergunta:** quanto custa treinar e usar um SVM com kernel RBF nos seus dados e no seu computador? Este notebook **mede** o custo (em vez de estimá-lo no escuro) para decidirmos o protocolo da busca de hiperparâmetros do SVM. Ele não escolhe hiperparâmetros: usa uma amostra da validação só para ver se o SVM funciona e como o normalizador se comporta.

| O que | Como |
|---|---|
| **Dados** | O mesmo ajuste (81.228 linhas, balanceado 1:1) e a mesma validação (60.154 linhas) do RF, sem os atributos de AUC alto. O tratamento dos ausentes é o do RF (mediana do treino + indicadores `_ausente`) |
| **Normalizador** | `Pipeline` com `StandardScaler` **só nas colunas numéricas**, ajustado **no próprio treino** de cada treino (sem vazamento na validação cruzada). Os indicadores `_ausente` ficam 0/1 (padronizá-los daria ~7 desvios a qualquer linha com valor ausente). Uma comparação rápida (**nenhum × padrão × robusto**) confere a escolha |
| **Custo × tamanho** | Treina com 10 mil, 20 mil, 40 mil e 81 mil linhas e mede tempo, CPU, RAM de pico e **vetores de suporte**; o tempo de previsão na validação dá o **custo por par (linha × vetor de suporte)**, que prevê o tempo de prever o teste |
| **Sensibilidade** | Como C e gamma mudam o tempo e o número de vetores de suporte (amostra média) |
| **Paralelismo** | Mede o ganho real de rodar vários treinos ao mesmo tempo e de prever em vários processos **neste computador**, com a memória livre consumida |
| **Estimativas** | A partir das medidas: tempo da busca (validação separada e CV de 5 dobras), do refit e da previsão do teste, no binário e (como referência) no multiclasse; com uma recomendação de protocolo |

## Proteções
Cada treino roda **num processo à parte, com limite de tempo** (`ORCAMENTO_FIT_S`, padrão 20 min): se passar, o processo é encerrado e o notebook segue. Um treino que, pela tendência, passaria do limite nem é iniciado. Se o seu SVM for lento, o notebook **para de aumentar o tamanho** e extrapola.

## O que o notebook grava
Na pasta `artefatos_modelos/svm_binario_calibracao/svm_cal_<data>`: as tabelas de cada medição (CSV), as estimativas, um gráfico (tempo e vetores de suporte × tamanho) e `calibracao_svm.json`.

## Antes de rodar
Os notebooks 02 a 05 precisam ter terminado. **Feche os outros notebooks** e programas pesados: o teste de paralelismo usa vários processos ao mesmo tempo (e o Meet atrapalha as medidas). Deixe o computador na tomada. Tempo estimado: de ~10 a ~40 minutos, conforme a velocidade do SVM nos seus dados.

In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

In [5]:
# ---------- ferramentas de métricas, pré-processamento e custo ----------
def _div(a, b): return float(a) / float(b) if b else float("nan")

def metricas(y, p, limiar=0.5, w=None):
    # Todas as métricas de uma vez. Com `w` (pesos), as contagens e as métricas que dependem da proporção ficam ponderadas.
    # O ROC-AUC é calculado sem pesos: ele só compara ataques com BENIGN e não depende da proporção.
    y = np.asarray(y).astype(int); p = np.asarray(p, dtype=float); pred = (p >= limiar).astype(int)
    w_ = np.ones(len(y)) if w is None else np.asarray(w, dtype=float)
    tp = float(w_[(y == 1) & (pred == 1)].sum()); tn = float(w_[(y == 0) & (pred == 0)].sum())
    fp = float(w_[(y == 0) & (pred == 1)].sum()); fn = float(w_[(y == 1) & (pred == 0)].sum())
    rec, esp, prec = _div(tp, tp + fn), _div(tn, tn + fp), _div(tp, tp + fp)
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    f1 = _div(2 * prec * rec, prec + rec) if (prec == prec and rec == rec) else float("nan")
    prec_b = _div(tn, tn + fn); f1_b = _div(2 * prec_b * esp, prec_b + esp) if (prec_b == prec_b and esp == esp) else float("nan")    # classe BENIGN como "positiva"
    return {"acuracia": _div(tp + tn, tp + tn + fp + fn), "acuracia_balanceada": (rec + esp) / 2, "precisao": prec, "recall": rec, "especificidade": esp,
            "taxa_falso_alarme": 1 - esp, "taxa_perda": 1 - rec, "f1": f1, "mcc": _div(tp * tn - fp * fn, den),
            "precisao_benign": prec_b, "f1_benign": f1_b, "f1_macro": (f1 + f1_b) / 2, "pr_auc_benign": float(average_precision_score(1 - y, 1 - p, sample_weight=w_)),
            "roc_auc": float(roc_auc_score(y, p)), "pr_auc": float(average_precision_score(y, p, sample_weight=w_)), "tp": tp, "fp": fp, "tn": tn, "fn": fn}

def recall_em_fpr(y, p, alvo):
    # Ponto de operação: o MAIOR recall com taxa de falso alarme <= alvo (limiar escolhido nestes dados).
    fpr, tpr, thr = roc_curve(np.asarray(y).astype(int), np.asarray(p, dtype=float))
    i = max(int(np.searchsorted(fpr, alvo, side="right")) - 1, 0)
    lim = float(thr[i]) if np.isfinite(thr[i]) else 1.0 + 1e-9
    return {"alvo_fpr": alvo, "limiar": lim, "recall": float(tpr[i]), "fpr": float(fpr[i])}

class Medidor:
    # Mede tempo de relógio, tempo de CPU (todos os núcleos do processo) e pico de RAM (RSS, amostrado a cada 50 ms) de um bloco `with`.
    def __init__(self, intervalo=0.05): self.intervalo = intervalo
    def __enter__(self):
        self.t0, self.c0 = time.perf_counter(), time.process_time(); self._ativo = psutil is not None
        if self._ativo:
            self._p = psutil.Process(); self.base = self._p.memory_info().rss; self.pico = self.base; self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set():
                    self.pico = max(self.pico, self._p.memory_info().rss); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        self.relogio, self.cpu = time.perf_counter() - self.t0, time.process_time() - self.c0
        if self._ativo:
            self._parar.set(); self._th.join(); self.pico = max(self.pico, self._p.memory_info().rss)
            self.pico_mib, self.delta_mib = self.pico / 2 ** 20, (self.pico - self.base) / 2 ** 20
        else:
            self.pico_mib = self.delta_mib = float("nan")

class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

def medir_inferencia(modelo, prep, X_modelo, X_raw, n_lat=1000, semente=42, repeticoes_lote=3):
    # Latência (1 fluxo por vez) e vazão (lote inteiro), com n_jobs = 1 e com o n_jobs original; com e sem o pré-processamento.
    rng = np.random.default_rng(semente); idx = rng.choice(len(X_modelo), size=min(n_lat, len(X_modelo)), replace=False)
    original = modelo.n_jobs; linhas = []
    for nj in dict.fromkeys([1, original]):
        modelo.set_params(n_jobs=nj)
        for i in idx[:20]: modelo.predict_proba(X_modelo[i:i + 1])                         # aquecimento
        t_mod, t_fim = [], []
        for i in idx:
            r = X_raw[i:i + 1]
            t0 = time.perf_counter(); m = prep.transformar(r); t1 = time.perf_counter(); modelo.predict_proba(m); t2 = time.perf_counter()
            t_mod.append(t2 - t1); t_fim.append(t2 - t0)
        l_mod, l_fim = np.percentile(t_mod, [50, 95, 99]) * 1000, np.percentile(t_fim, [50, 95, 99]) * 1000
        v_mod, v_fim = [], []
        for _ in range(repeticoes_lote):
            t0 = time.perf_counter(); modelo.predict_proba(X_modelo); v_mod.append(len(X_modelo) / (time.perf_counter() - t0))
            t0 = time.perf_counter(); modelo.predict_proba(prep.transformar(X_raw)); v_fim.append(len(X_raw) / (time.perf_counter() - t0))
        linhas.append({"n_jobs": nj, "n_latencia": len(idx), "latencia_modelo_p50_ms": l_mod[0], "latencia_modelo_p95_ms": l_mod[1], "latencia_modelo_p99_ms": l_mod[2],
                       "latencia_fim_a_fim_p50_ms": l_fim[0], "latencia_fim_a_fim_p95_ms": l_fim[1], "latencia_fim_a_fim_p99_ms": l_fim[2],
                       "vazao_modelo_fluxos_s": float(np.median(v_mod)), "vazao_fim_a_fim_fluxos_s": float(np.median(v_fim)), "n_lote": len(X_modelo)})
    modelo.set_params(n_jobs=original)
    return linhas

def tamanho_e_nos(modelo, caminho, compressao):
    joblib.dump(modelo, caminho, compress=compressao)
    return {"arquivo_mib": os.path.getsize(caminho) / 2 ** 20, "n_arvores": len(modelo.estimators_), "nos_total": int(sum(e.tree_.node_count for e in modelo.estimators_)),
            "profundidade_media": float(np.mean([e.get_depth() for e in modelo.estimators_])), "profundidade_max": int(max(e.get_depth() for e in modelo.estimators_))}

# autoteste das ferramentas (não toca no servidor)
_y = np.array([0, 0, 0, 0, 1, 1, 1, 1]); _p = np.array([.1, .2, .3, .6, .4, .7, .8, .9]); _m = metricas(_y, _p, 0.5)
assert (_m["tp"], _m["fp"], _m["tn"], _m["fn"]) == (3, 1, 3, 1) and abs(_m["mcc"] - 0.5) < 1e-12 and abs(_m["acuracia"] - 0.75) < 1e-12, _m
_o = recall_em_fpr(_y, _p, 0.25); assert abs(_o["recall"] - 1.0) < 1e-12 and _o["fpr"] <= 0.25 and abs(_o["limiar"] - 0.4) < 1e-12, _o
_m2 = metricas(_y, _p, 0.5); assert abs(_m2["precisao_benign"] - 0.75) < 1e-12 and abs(_m2["f1_benign"] - 0.75) < 1e-12 and abs(_m2["f1_macro"] - 0.75) < 1e-12, _m2
_o0 = recall_em_fpr(_y, _p, 0.0); assert abs(_o0["recall"] - 0.75) < 1e-12 and _o0["fpr"] == 0.0, _o0
_d = distribuir(100, {"a": 1000, "b": 1000, "c": 1000, "d": 10}, 0.0, {"a": 1000, "b": 1000, "c": 1000, "d": 10}); assert _d["d"] == 10 and sum(_d.values()) == 100, _d
print("Autoteste das ferramentas OK | sklearn", sklearn.__version__, "| psutil", "ausente" if psutil is None else psutil.__version__)

Autoteste das ferramentas OK | sklearn 1.9.1 | psutil 7.2.2


### Barra de progresso
`Barra` mostra o andamento (percentual, tempo decorrido e tempo restante estimado) das etapas com várias medições.

In [6]:
import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")

_b = Barra(2, "autoteste"); _b.avancar(2); assert fmt_tempo(3725) == "1h 02min"
print("Barra de progresso pronta")

Barra de progresso pronta


## 1. Parâmetros, dados e normalizador
Lê o ajuste e a validação (as mesmas views do RF, ordenadas por `id_linha`), junta o tipo de ataque (para as amostras serem **estratificadas por tipo**) e define o `Pipeline` do SVM. `novo_pipeline(C, gamma, escala)` cria o modelo: `escala = "padrao"` (o escolhido), `"robusto"` (mediana e intervalo interquartil, menos sensível a valores extremos) ou `"nenhum"` (só para comparar).

In [7]:
# ---------- 1.1 Parâmetros, dados e normalizador ----------
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from joblib import Parallel, delayed
from joblib.externals.loky import ProcessPoolExecutor as LokyPool
import concurrent.futures

TAMANHOS = tuple(globals().get("TAMANHOS", (10_000, 20_000, 40_000, None)))          # None = o ajuste inteiro
PROCESSOS = tuple(globals().get("PROCESSOS", (1, 2, 4, 6, 8)))                       # processos simultâneos a testar
C_VALORES = tuple(globals().get("C_VALORES", (0.1, 1.0, 10.0, 100.0)))
GAMMA_FATORES = tuple(globals().get("GAMMA_FATORES", (0.25, 1.0, 4.0)))              # múltiplos do gamma "scale"
CACHE_MB = globals().get("CACHE_MB", 200)                                            # cache de kernel de cada SVM (por processo)
ORCAMENTO_FIT_S = globals().get("ORCAMENTO_FIT_S", 1200)                             # nenhum treino pode passar disto (o processo é encerrado)
ORCAMENTO_SENS_S = globals().get("ORCAMENTO_SENS_S", 90)                             # as comparações usam o maior tamanho cujo treino levou até isto
LIMITE_BUSCA_H = globals().get("LIMITE_BUSCA_H", 6.0); LIMITE_TESTE_H = globals().get("LIMITE_TESTE_H", 1.5)    # para a recomendação (em horas)
SEMENTE = globals().get("SEMENTE", 42)
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
PASTA = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos" / "svm_binario_calibracao" / f"svm_cal_{RUN_ID}"; PASTA.mkdir(parents=True, exist_ok=True)

V_AJUSTE, V_VALID, V_REFIT, SEL_TAB = "prep_vw_bal_bin_treino", "prep_vw_bal_bin_validacao", "prep_vw_bal_bin_refit", "prep_bal_bin_selecao"
VIEW_TESTE = "prep_vw_rf_bin_teste_sem_sobrep"
faltam = [t for t in ("prep_final_treino", V_AJUSTE, V_VALID, V_REFIT, SEL_TAB, VIEW_TESTE, "prep_views_rf_binario") if not existe(t)]
assert not faltam, f"Rode antes os notebooks 02 a 05: faltam {faltam}"
aud = consultar_num("SELECT posicao, feature, situacao, mediana_imputacao, tem_indicador, versao FROM prep_views_rf_binario ORDER BY posicao"); versao_views = str(aud["versao"].iloc[0])
USADAS, RETIRADAS = aud[aud.situacao == "usada"], aud[aud.situacao != "usada"].feature.tolist()
MEDIANAS = {r.feature: float(r.mediana_imputacao) for r in USADAS.itertuples() if int(r.tem_indicador) == 1}
prep = PreprocessadorRF(USADAS.feature.tolist(), MEDIANAS, RETIRADAS, versao_views); CM = prep.colunas_modelo
assert CM == [c for c in colunas_view(V_AJUSTE) if c not in ("id_linha", "y")], "a ordem das colunas da view difere da do pré-processamento"
NUM, IND = list(range(len(prep.features))), list(range(len(prep.features), len(CM)))           # as colunas numéricas vêm primeiro; os indicadores _ausente, no fim

def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
aj, va = ordenar(ler_view(V_AJUSTE)), ordenar(ler_view(V_VALID))
X_aj, y_aj, X_va, y_va = aj[CM].to_numpy(float), aj["y"].to_numpy(int), va[CM].to_numpy(float), va["y"].to_numpy(int)
for nome, X in (("ajuste", X_aj), ("validação", X_va)): assert np.isfinite(X).all(), f"valores não finitos em {nome}"
nome_tipo = lambda t: t.replace("DrDoS_", "", 1) if t.startswith("DrDoS_") else t
sel = consultar(f"SELECT toString(id_linha) AS id_linha, papel, tipo FROM {SEL_TAB}"); sel["tipo"] = sel["tipo"].map(nome_tipo)
tipo_aj = aj[["id_linha"]].merge(sel[sel.papel == "treino"][["id_linha", "tipo"]], on="id_linha", how="left")["tipo"]; assert tipo_aj.notna().all()
g_aj = tipo_aj.astype("category").cat.codes.to_numpy()
N_AJ, N_VA, N_RF, N_TESTE = len(X_aj), len(X_va), n_linhas(V_REFIT), n_linhas(VIEW_TESTE)

def novo_pipeline(C=1.0, gamma="scale", escala="padrao", cache=None):
    # Pipeline do SVM: normaliza (média e desvio do próprio treino) só as colunas numéricas; os indicadores _ausente ficam 0/1; depois o SVM RBF.
    if escala == "nenhum": pre = "passthrough"
    else:
        esc = StandardScaler() if escala == "padrao" else RobustScaler()
        pre = ColumnTransformer([("num", esc, NUM)] + ([("ind", "passthrough", IND)] if IND else []))
    return Pipeline([("escala", pre), ("svm", SVC(kernel="rbf", C=C, gamma=gamma, cache_size=cache or CACHE_MB))])

def amostra(n):
    # n linhas do ajuste, estratificadas por tipo (BENIGN continua 50%); None = tudo
    if n is None or n >= N_AJ: return np.arange(N_AJ)
    idx, _ = train_test_split(np.arange(N_AJ), train_size=n, stratify=g_aj, random_state=SEMENTE); return np.sort(idx)

print(f"Pasta da calibração: {PASTA}")
print(f"Ajuste: {N_AJ:,} linhas ({int((y_aj == 0).sum()):,} BENIGN) | validação: {N_VA:,} ({int((y_va == 0).sum()):,} BENIGN) | refit: {N_RF:,} | teste: {N_TESTE:,}")
print(f"Atributos: {len(CM)} = {len(NUM)} numéricos (padronizados) + {len(IND)} indicadores _ausente (ficam 0/1)")
print(f"Computador: {os.cpu_count()} núcleos lógicos" + (f" ({psutil.cpu_count(logical=False)} físicos)" if psutil else "") + (f", {psutil.virtual_memory().total / 2 ** 30:.1f} GiB de RAM, {psutil.virtual_memory().available / 2 ** 30:.1f} GiB livres agora" if psutil else ""))
# conferência do normalizador: ajustado só no treino, numéricos com média 0 e desvio 1, indicadores intactos
_p = novo_pipeline().fit(X_aj[amostra(5000)], y_aj[amostra(5000)]); _Xt = _p.named_steps["escala"].transform(X_aj[amostra(5000)])
assert np.allclose(_Xt[:, NUM].mean(0), 0, atol=1e-9) and np.allclose(_Xt[:, NUM].std(0), 1, atol=1e-6) or (_Xt[:, NUM].std(0) == 0).any(), "o normalizador não deixou as colunas numéricas com média 0 e desvio 1"
assert (not IND) or np.array_equal(_Xt[:, IND], X_aj[amostra(5000)][:, IND]), "os indicadores _ausente foram alterados pelo normalizador"
print("✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)")

Pasta da calibração: C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_calibracao\svm_cal_20261024_224947
Ajuste: 81,228 linhas (40,614 BENIGN) | validação: 60,154 (10,154 BENIGN) | refit: 101,536 | teste: 7,219,189
Atributos: 43 = 42 numéricos (padronizados) + 1 indicadores _ausente (ficam 0/1)
Computador: 12 núcleos lógicos (10 físicos), 7.7 GiB de RAM, 0.8 GiB livres agora
✔ Normalizador conferido: colunas numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos (0/1)


### Diagnóstico das caudas
O padronizador usa a média e o desvio, que são sensíveis a valores extremos: uma coluna com uns poucos valores gigantes fica com um desvio enorme, e quase todas as linhas ficam coladas perto de 0 nela. A tabela mostra, no ajuste inteiro, as colunas com os maiores desvios em relação ao desvio-padrão (`|z|` máximo) e quantas linhas passam de 5 desvios. Se muitas colunas tiverem `|z|` máximo muito alto, o normalizador **robusto** (comparado mais adiante) pode ser melhor.

In [8]:
# ---------- 1.2 Diagnóstico das caudas ----------
Z = StandardScaler().fit_transform(X_aj[:, NUM]); az = np.abs(Z)
diag = pd.DataFrame({"atributo": [prep.features[i] for i in NUM], "|z| máximo": az.max(0), "% linhas com |z| > 5": 100 * (az > 5).mean(0), "% linhas com |z| > 20": 100 * (az > 20).mean(0)}).sort_values("|z| máximo", ascending=False).reset_index(drop=True)
print(f"Colunas numéricas com |z| máximo > 50: {int((diag['|z| máximo'] > 50).sum())} de {len(diag)} | com mais de 1% das linhas acima de 5 desvios: {int((diag['% linhas com |z| > 5'] > 1).sum())}")
display(diag.head(10).round(3)); del Z, az

Colunas numéricas com |z| máximo > 50: 9 de 42 | com mais de 1% das linhas acima de 5 desvios: 7


,atributo,|z| máximo,% linhas com |z| > 5,% linhas com |z| > 20
0,total_length_of_fwd_packets,279.768,0.001,0.001
1,act_data_pkt_fwd,171.839,1.141,0.001
2,total_length_of_bwd_packets,118.667,0.057,0.031
3,packet_length_variance,108.805,0.028,0.023
4,flow_iat_min,100.699,0.196,0.032
5,fwd_packet_length_max,63.332,0.261,0.001
6,total_fwd_packets,61.943,0.030,0.030
7,active_std,52.319,0.614,0.031
8,fwd_header_length,51.241,0.078,0.052
9,bwd_packet_length_max,46.193,0.027,0.027


## 2. Ferramentas da calibração
- **`_fit_worker`**: treina um SVM (`Pipeline`) numa amostra e, no mesmo processo, mede tempo de relógio e de CPU, pico de RAM, vetores de suporte, o tempo de transformar e de prever a validação (daí o **custo por par**, em ns) e, de forma informativa, ROC-AUC, recall e falso alarme da validação (limiar 0 do SVM).
- **`executar_com_limite`**: roda uma função **num processo separado com limite de tempo**; se passar, o processo é encerrado e o notebook segue.
- **`MedidorSistema`**: mede quanto a RAM livre do computador caiu durante um bloco (soma de todos os processos).
- **`ajustar_potencia`**: ajusta `tempo = a × n^b` (e o mesmo para os vetores de suporte), para estimar tamanhos que não foram medidos.

In [9]:
class MedidorSistema:
    # Quanto a RAM livre do computador (todos os programas) caiu durante um bloco `with`.
    def __init__(self, intervalo=0.2): self.intervalo = intervalo
    def __enter__(self):
        self.inicio = psutil.virtual_memory().available if psutil else float("nan"); self.minimo = self.inicio
        if psutil:
            self._parar = threading.Event()
            def amostrar():
                while not self._parar.is_set(): self.minimo = min(self.minimo, psutil.virtual_memory().available); self._parar.wait(self.intervalo)
            self._th = threading.Thread(target=amostrar, daemon=True); self._th.start()
        return self
    def __exit__(self, *a):
        if psutil: self._parar.set(); self._th.join(); self.minimo = min(self.minimo, psutil.virtual_memory().available)
        self.livre_inicial_gib = self.inicio / 2 ** 30; self.queda_gib = (self.inicio - self.minimo) / 2 ** 30

def _fit_worker(X, y, Xv, yv, C, gamma, escala, devolver_modelo=False):
    pipe = novo_pipeline(C, gamma, escala)
    with Medidor() as m: pipe.fit(X, y)
    svm, pre = pipe.named_steps["svm"], pipe.named_steps["escala"]
    t0 = time.perf_counter(); Xt = Xv if isinstance(pre, str) else pre.transform(Xv); t_tr = time.perf_counter() - t0
    t0 = time.perf_counter(); sc = svm.decision_function(Xt); t_sv = time.perf_counter() - t0
    nsv = int(svm.n_support_.sum())
    r = {"n": int(len(y)), "C": float(C), "gamma": float(getattr(svm, "_gamma", float("nan"))), "escala": escala, "tempo_treino_s": m.relogio, "cpu_s": m.cpu, "pico_ram_mib": m.pico_mib, "acrescimo_ram_mib": m.delta_mib,
         "n_sv": nsv, "frac_sv": nsv / len(y), "tempo_transformacao_val_s": t_tr, "tempo_svm_val_s": t_sv, "ns_por_par": t_sv / (len(Xv) * max(nsv, 1)) * 1e9,
         "roc_auc_val": float(roc_auc_score(yv, sc)), "recall_0": float((sc[yv == 1] > 0).mean()), "fpr_0": float((sc[yv == 0] > 0).mean())}
    return (r, pipe if devolver_modelo else None)

def executar_com_limite(fn, args, limite_s, barra=None, rotulo=""):
    # Um processo NOVO e independente por chamada (não compartilha nada com o Parallel do joblib); se passar do limite, é encerrado.
    # Espera em fatias de 0,5 s: o botão "interromper" do Jupyter funciona (também no Windows) e a barra mostra o tempo decorrido.
    ex = LokyPool(max_workers=1); fut = ex.submit(fn, *args); t0 = time.perf_counter()
    try:
        while True:
            decorrido = time.perf_counter() - t0
            if decorrido >= limite_s: ex.shutdown(wait=True, kill_workers=True); return None
            if barra is not None: barra._extra = f"{rotulo} | em andamento há {fmt_tempo(decorrido)} (limite {fmt_tempo(limite_s)})"; barra._mostrar()
            try: res = fut.result(timeout=min(0.5, limite_s - decorrido))
            except (concurrent.futures.TimeoutError, TimeoutError): continue
            ex.shutdown(wait=True); return res
    except BaseException:
        ex.shutdown(wait=True, kill_workers=True); raise

def ajustar_potencia(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float); ok_ = (x > 0) & (y > 0); b, loga = np.polyfit(np.log(x[ok_]), np.log(y[ok_]), 1)
    return (lambda n: float(np.exp(loga) * float(n) ** b)), float(b)
print("Ferramentas da calibração prontas")

Ferramentas da calibração prontas


## 3. Custo × tamanho
Treina o SVM (`C = 1`, `gamma = "scale"`, normalizador padrão) com amostras crescentes do ajuste (estratificadas por tipo; BENIGN continua 50%) e mede o custo de cada treino e da previsão na validação. Se o treino de um tamanho for **previsto** para passar do limite de tempo (pela tendência dos tamanhos menores), ele não é rodado; se for rodado e passar, é encerrado. **ROC-AUC, recall e falso alarme são só informativos** (limiar 0 do SVM): nada é escolhido com eles aqui.

In [10]:
# ---------- 3.1 Custo × tamanho ----------
res_tam, MODELO = [], None
barra = Barra(len(TAMANHOS), "Custo × tamanho")
for n in TAMANHOS:
    n_real = N_AJ if n is None else min(n, N_AJ)
    if len(res_tam) >= 2:
        (n1, t1), (n2, t2) = [(r["n"], r["tempo_treino_s"]) for r in res_tam[-2:]]; b = np.log(t2 / t1) / np.log(n2 / n1) if (t1 > 0 and t2 > 0 and n2 > n1) else 2.0; prev = t2 * (n_real / n2) ** max(b, 1.0)
        if prev > ORCAMENTO_FIT_S:
            print(f"  ⚠ o treino com {n_real:,} linhas é previsto em ~{fmt_tempo(prev)}, acima do limite ({fmt_tempo(ORCAMENTO_FIT_S)}): não será rodado (as estimativas usarão a tendência dos tamanhos menores)"); barra.avancar(1, f"{n_real:,} linhas: pulado"); continue
    idx = amostra(n_real); barra.avancar(0, f"treinando com {n_real:,} linhas (limite {fmt_tempo(ORCAMENTO_FIT_S)})...")
    out = executar_com_limite(_fit_worker, (X_aj[idx], y_aj[idx], X_va, y_va, 1.0, "scale", "padrao", True), ORCAMENTO_FIT_S, barra, f"treinando com {n_real:,} linhas")
    if out is None:
        print(f"  ⚠ o treino com {n_real:,} linhas passou de {fmt_tempo(ORCAMENTO_FIT_S)} e foi encerrado"); barra.avancar(1, "passou do limite"); break
    r, pipe = out; res_tam.append(r); MODELO = pipe
    barra.avancar(1, f"{n_real:,} linhas: treino {fmt_tempo(r['tempo_treino_s'])}, {r['n_sv']:,} vetores de suporte ({100 * r['frac_sv']:.0f}%), ROC-AUC {r['roc_auc_val']:.5f}")
barra.fechar(); assert res_tam, "nenhum treino foi concluído dentro do limite de tempo: aumente ORCAMENTO_FIT_S ou diminua TAMANHOS"
tab_tam = pd.DataFrame(res_tam); print("\n== Custo × tamanho (validação com", f"{N_VA:,} linhas) =="); display(tab_tam[["n", "tempo_treino_s", "cpu_s", "pico_ram_mib", "n_sv", "frac_sv", "tempo_svm_val_s", "ns_por_par", "roc_auc_val", "recall_0", "fpr_0"]].round(4))
N_MAX_MEDIDO = int(tab_tam["n"].max())
if len(res_tam) >= 2:
    f_t, b_t = ajustar_potencia(tab_tam["n"], tab_tam["tempo_treino_s"]); f_sv, b_sv = ajustar_potencia(tab_tam["n"], tab_tam["n_sv"]); SUPOSTO = False
else:
    r0 = res_tam[0]; b_t, b_sv = 2.0, 1.0; f_t = lambda n, r0=r0: r0["tempo_treino_s"] * (n / r0["n"]) ** 2.0; f_sv = lambda n, r0=r0: r0["n_sv"] * (n / r0["n"]); SUPOSTO = True
NS_PAR = float(tab_tam["ns_por_par"].iloc[-min(2, len(tab_tam)):].mean())
print(f"\nO tempo de treino cresce como n^{b_t:.2f} e os vetores de suporte como n^{b_sv:.2f}" + ("  (⚠ só um tamanho medido: expoentes SUPOSTOS 2 e 1)" if SUPOSTO else f"  (ajuste com {len(res_tam)} tamanhos)") + f" | custo de previsão: ~{NS_PAR:.0f} ns por par (linha × vetor de suporte)")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 2, figsize=(11, 4)); xs = np.linspace(tab_tam["n"].min(), max(N_RF, N_MAX_MEDIDO), 50)
    ax[0].loglog(tab_tam["n"], tab_tam["tempo_treino_s"], "o", label="medido"); ax[0].loglog(xs, [f_t(v) for v in xs], "--", label=f"tendência n^{b_t:.2f}"); ax[0].set_xlabel("linhas de treino"); ax[0].set_ylabel("tempo de treino (s)"); ax[0].legend(); ax[0].grid(alpha=.3, which="both")
    ax[1].loglog(tab_tam["n"], tab_tam["n_sv"], "o", label="medido"); ax[1].loglog(xs, [f_sv(v) for v in xs], "--", label=f"tendência n^{b_sv:.2f}"); ax[1].set_xlabel("linhas de treino"); ax[1].set_ylabel("vetores de suporte"); ax[1].legend(); ax[1].grid(alpha=.3, which="both")
    fig.tight_layout(); fig.savefig(PASTA / "custo_por_tamanho.png", dpi=130); plt.close(fig)
except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")

Custo × tamanho: concluído em 7min 05s

== Custo × tamanho (validação com 60,154 linhas) ==


,n,tempo_treino_s,cpu_s,pico_ram_mib,n_sv,frac_sv,tempo_svm_val_s,ns_por_par,roc_auc_val,recall_0,fpr_0
0,10000,0.6633,0.6562,207.0742,552,0.0552,10.0460,302.5451,0.9992,0.9918,0.0044
1,20000,1.7053,1.7344,258.2500,808,0.0404,8.4586,174.0288,0.9990,0.9929,0.0043
2,40000,6.5902,6.5000,420.9922,1252,0.0313,13.8135,183.4144,0.9989,0.9937,0.0043
3,81228,341.1020,335.2969,492.1914,2152,0.0265,28.4965,220.1325,0.9993,0.9941,0.0042



O tempo de treino cresce como n^2.88 e os vetores de suporte como n^0.65  (ajuste com 4 tamanhos) | custo de previsão: ~202 ns por par (linha × vetor de suporte)


## 4. Sensibilidade a C e gamma
Com uma amostra média (o maior tamanho cujo treino levou até `ORCAMENTO_SENS_S`), treina o SVM com vários valores de **C** (rigidez da margem: valores altos tendem a mais vetores de suporte e treino mais lento) e de **gamma** (alcance do kernel, em múltiplos do `scale`). Mostra como o **tempo** e o número de **vetores de suporte** mudam: é isso que define o custo da grade. Os ROC-AUC são só informativos (a escolha dos hiperparâmetros será feita na busca, pelo protocolo). Um treino que passar do limite é encerrado e marcado.

In [11]:
# ---------- 4.1 Sensibilidade a C e gamma ----------
cand = [r["n"] for r in res_tam if r["tempo_treino_s"] <= ORCAMENTO_SENS_S]; N_SENS = max(cand) if cand else int(tab_tam["n"].min())
base = next(r for r in res_tam if r["n"] == N_SENS); gamma0 = base["gamma"]; idx_s = amostra(N_SENS); Xs, ys = X_aj[idx_s], y_aj[idx_s]
combos = [(C, 1.0) for C in C_VALORES if not (C == 1.0)] + [(1.0, f) for f in GAMMA_FATORES if f != 1.0]
print(f"Amostra da sensibilidade: {N_SENS:,} linhas | gamma 'scale' = {gamma0:.5f} | base (C = 1, gamma = scale): treino {fmt_tempo(base['tempo_treino_s'])}, {base['n_sv']:,} vetores de suporte")
linhas_sens = [{**base, "gamma_fator": 1.0, "concluiu": True}]; LIMITE_SENS_S = min(ORCAMENTO_FIT_S, max(120, 10 * base["tempo_treino_s"]))          # um treino 10× mais lento que o da base já é caro demais para a grade
print(f"Limite por treino nesta seção: {fmt_tempo(LIMITE_SENS_S)} (10× o da base, no mínimo 2 min); quem passar é encerrado e marcado")
barra = Barra(len(combos), "Sensibilidade a C e gamma")
for C, f in combos:
    barra.avancar(0, f"C = {C:g}, gamma = {f:g}×scale...")
    out = executar_com_limite(_fit_worker, (Xs, ys, X_va, y_va, C, gamma0 * f, "padrao", False), LIMITE_SENS_S, barra, f"C = {C:g}, gamma = {f:g}×scale")
    if out is None: linhas_sens.append({"n": N_SENS, "C": C, "gamma": gamma0 * f, "gamma_fator": f, "escala": "padrao", "tempo_treino_s": float("nan"), "concluiu": False}); barra.avancar(1, f"C = {C:g}, gamma = {f:g}×scale: PASSOU DO LIMITE"); continue
    r = out[0]; linhas_sens.append({**r, "gamma_fator": f, "concluiu": True}); barra.avancar(1, f"C = {C:g}, gamma = {f:g}×scale: {fmt_tempo(r['tempo_treino_s'])}, {r['n_sv']:,} vetores de suporte")
barra.fechar(); tab_sens = pd.DataFrame(linhas_sens).sort_values(["C", "gamma_fator"]).reset_index(drop=True)
print("\n== Sensibilidade a C e gamma (informativa) =="); display(tab_sens[["C", "gamma_fator", "gamma", "tempo_treino_s", "n_sv", "frac_sv", "ns_por_par", "roc_auc_val", "recall_0", "fpr_0", "concluiu"]].round(5))
t_min, t_max = tab_sens["tempo_treino_s"].min(), tab_sens["tempo_treino_s"].max(); FATOR_GRADE = float(t_max / t_min) if t_min > 0 else float("nan")
print(f"O tempo de treino variou {FATOR_GRADE:.1f}× entre as combinações (de {fmt_tempo(t_min)} a {fmt_tempo(t_max)}): a grade fica mais cara nos cantos de C e gamma mais altos.")

Amostra da sensibilidade: 40,000 linhas | gamma 'scale' = 0.02380 | base (C = 1, gamma = scale): treino 7s, 1,252 vetores de suporte
Limite por treino nesta seção: 2min 00s (10× o da base, no mínimo 2 min); quem passar é encerrado e marcado


Sensibilidade a C e gamma: concluído em 2min 49s

== Sensibilidade a C e gamma (informativa) ==


,C,gamma_fator,gamma,tempo_treino_s,n_sv,frac_sv,ns_por_par,roc_auc_val,recall_0,fpr_0,concluiu
0,0.1,1.00,0.02380,16.05748,2394,0.05985,201.85930,0.99903,0.98988,0.00423,True
1,1.0,0.25,0.00595,9.90391,1589,0.03973,190.07272,0.99890,0.99280,0.00453,True
2,1.0,1.00,0.02380,6.59019,1252,0.03130,183.41443,0.99895,0.99368,0.00433,True
3,1.0,4.00,0.09521,13.36025,1752,0.04380,238.02767,0.99927,0.99350,0.00414,True
4,10.0,1.00,0.02380,8.59486,993,0.02482,241.96889,0.99938,0.99432,0.00414,True
5,100.0,1.00,0.02380,7.14999,824,0.02060,205.97800,0.99915,0.99510,0.00374,True


O tempo de treino variou 2.4× entre as combinações (de 7s a 16s): a grade fica mais cara nos cantos de C e gamma mais altos.


## 5. Comparação rápida de normalizadores
Na mesma amostra e com `C = 1`, `gamma = "scale"`: **nenhum** (dados na escala original), **padrão** (média 0, desvio 1; o escolhido) e **robusto** (mediana e intervalo interquartil). Confere se normalizar importa e se o robusto vale a pena nos seus dados. **Cada treino aqui tem um limite curto** (5× o tempo do padrão, no mínimo 1 min): sem normalizar o SVM costuma ser muito lento ou nem convergir, e isso por si só já é um resultado (aparece como "passou do limite"). Os números de ROC-AUC são **da validação** (informativos, uma só amostra e um só ponto da grade): servem para ver ordens de grandeza, não para decidir sozinhos.

In [12]:
# ---------- 5.1 Normalizadores ----------
LIMITE_NORM_S = min(ORCAMENTO_FIT_S, globals().get("ORCAMENTO_NORM_S") or max(60, 5 * base["tempo_treino_s"]))     # 5× o treino com o normalizador padrão (no mínimo 1 min)
print(f"Limite por treino nesta seção: {fmt_tempo(LIMITE_NORM_S)} (5× o do normalizador padrão). SEM normalizar o SVM costuma ser muito lento ou nem converge: se passar do limite, é encerrado e marcado.")
linhas_norm = [{**base, "normalizador": "padrão"}]; barra = Barra(2, "Normalizadores")
for esc, nome in (("robusto", "robusto"), ("nenhum", "nenhum")):
    barra.avancar(0, f"{nome}...")
    out = executar_com_limite(_fit_worker, (Xs, ys, X_va, y_va, 1.0, "scale", esc, False), LIMITE_NORM_S, barra, f"normalizador {nome}")
    if out is None: linhas_norm.append({"n": N_SENS, "normalizador": nome, "tempo_treino_s": float("nan")}); barra.avancar(1, f"{nome}: PASSOU DO LIMITE ({fmt_tempo(LIMITE_NORM_S)}) e foi encerrado"); continue
    linhas_norm.append({**out[0], "normalizador": nome}); barra.avancar(1, f"{nome}: {fmt_tempo(out[0]['tempo_treino_s'])}, ROC-AUC {out[0]['roc_auc_val']:.5f}")
barra.fechar(); tab_norm = pd.DataFrame(linhas_norm).set_index("normalizador").loc[["nenhum", "padrão", "robusto"]].reset_index()
print(f"\n== Normalizadores (amostra de {N_SENS:,} linhas; C = 1, gamma = scale) ==")
display(tab_norm[[c for c in ("normalizador", "tempo_treino_s", "n_sv", "frac_sv", "roc_auc_val", "recall_0", "fpr_0") if c in tab_norm.columns]].round(5))
pad_, rob_, nen_ = (tab_norm[tab_norm.normalizador == k].iloc[0] for k in ("padrão", "robusto", "nenhum")); valido = lambda r: bool(r["roc_auc_val"] == r["roc_auc_val"])
if valido(rob_) and valido(nen_): print(f"ROC-AUC na validação: nenhum {nen_['roc_auc_val']:.5f} | padrão {pad_['roc_auc_val']:.5f} | robusto {rob_['roc_auc_val']:.5f}")
if not valido(nen_): print(f"→ SEM normalizar, o treino passou de {fmt_tempo(LIMITE_NORM_S)} (o padrão levou {fmt_tempo(pad_['tempo_treino_s'])}): o SVM precisa de normalização nestes dados, como esperado.")
else: print("→ " + ("sem normalizar o resultado é pior, como esperado." if nen_["roc_auc_val"] < pad_["roc_auc_val"] - 0.002 else "sem normalizar a diferença foi pequena nesta amostra."))
if not valido(rob_): print(f"→ o normalizador robusto passou do limite: fica o padrão.")
else: print("→ " + ("o robusto ficou claramente acima do padrão: vale discutir trocá-lo antes da busca." if rob_["roc_auc_val"] > pad_["roc_auc_val"] + 0.002 else "o padrão não ficou atrás do robusto de forma relevante: mantemos o padrão."))

Limite por treino nesta seção: 1min 00s (5× o do normalizador padrão). SEM normalizar o SVM costuma ser muito lento ou nem converge: se passar do limite, é encerrado e marcado.


Normalizadores: concluído em 2min 00s

== Normalizadores (amostra de 40,000 linhas; C = 1, gamma = scale) ==


,normalizador,tempo_treino_s,n_sv,frac_sv,roc_auc_val,recall_0,fpr_0
0,nenhum,NaN,NaN,NaN,NaN,NaN,NaN
1,padrão,6.59019,1252.0,0.0313,0.99895,0.99368,0.00433
2,robusto,NaN,NaN,NaN,NaN,NaN,NaN


→ SEM normalizar, o treino passou de 1min 00s (o padrão levou 7s): o SVM precisa de normalização nestes dados, como esperado.
→ o normalizador robusto passou do limite: fica o padrão.


## 6. Paralelismo: treinar e prever em vários processos
O SVM do scikit-learn usa **um núcleo** por treino. O ganho vem de rodar **vários treinos ao mesmo tempo** (as combinações da grade e as dobras são independentes) e de **prever o teste em pedaços, em vários processos**. Aqui medimos o ganho real neste computador, e a RAM que o conjunto consome (soma de todos os programas): rodar mais processos do que a memória ou os núcleos aguentam deixa tudo mais lento.
- **Ganho** = quantos treinos por minuto em relação a 1 processo; **eficiência** = ganho ÷ número de processos (1,0 = perfeito). Um processo a mais só vale a pena se a eficiência continuar razoável (≥ 0,5) e sobrar memória.
- O número recomendado (`K_REC`) é o de maior ganho com eficiência ≥ 0,5 e queda de RAM de até 70% da memória livre.
- A medição da previsão inclui o custo de enviar o modelo aos processos (conservador). Mede também se as previsões em paralelo dão **exatamente os mesmos escores** que a sequencial.

In [13]:
# ---------- 6.1 Treinos simultâneos ----------
Xvp, yvp = X_va[:2000], y_va[:2000]; assert len(np.unique(yvp)) == 2
IGNORAR_NUCLEOS = globals().get("IGNORAR_NUCLEOS", False)                                  # só para testes
procs = sorted({1} | {k for k in PROCESSOS if IGNORAR_NUCLEOS or k <= (os.cpu_count() or 1)}); linhas_par = []; barra = Barra(len(procs), f"Treinos simultâneos ({N_SENS:,} linhas cada)")
for k in procs:
    if psutil and k > 1:
        precisa = k * (base["pico_ram_mib"] + CACHE_MB) * 1.3 / 1024; livre = psutil.virtual_memory().available / 2 ** 30
        if precisa > 0.8 * livre: print(f"  ⚠ {k} processos precisariam de ~{precisa:.1f} GiB e só há {livre:.1f} GiB livres: pulado"); barra.avancar(1, f"{k} processos: pulado (pouca memória)"); continue
    if k > 1: Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k))                  # aquece os processos (a partida deles não entra na medida)
    barra.avancar(0, f"{k} treino(s) ao mesmo tempo...")
    with MedidorSistema() as ms:
        t0 = time.perf_counter(); outs = Parallel(n_jobs=k)(delayed(_fit_worker)(Xs, ys, Xvp, yvp, 1.0, "scale", "padrao", False) for _ in range(k)); wall = time.perf_counter() - t0
    rs = [o[0] for o in outs]; linhas_par.append({"processos": k, "tempo_total_s": wall, "tempo_medio_por_treino_s": float(np.mean([r["tempo_treino_s"] for r in rs])), "n_sv_igual_em_todos": len({r["n_sv"] for r in rs}) == 1,
                                                    "n_sv": rs[0]["n_sv"], "ram_livre_inicial_gib": ms.livre_inicial_gib, "queda_ram_gib": ms.queda_gib})
    barra.avancar(1, f"{k} processo(s): {fmt_tempo(wall)} no total")
barra.fechar(); tab_par = pd.DataFrame(linhas_par); v1 = 1 / float(tab_par.loc[tab_par.processos == 1, "tempo_total_s"].iloc[0])
tab_par["treinos_por_min"] = 60 * tab_par.processos / tab_par.tempo_total_s; tab_par["ganho"] = (tab_par.processos / tab_par.tempo_total_s) / v1; tab_par["eficiencia"] = tab_par.ganho / tab_par.processos
bons = tab_par[(tab_par.queda_ram_gib <= 0.7 * tab_par.ram_livre_inicial_gib) & (tab_par.eficiencia >= 0.5)]; K_REC = int(bons.loc[bons.ganho.idxmax(), "processos"]) if (len(bons) and bons.ganho.max() >= 1.2) else 1          # só recomenda mais de 1 processo se o ganho for real (≥ 1,2×)
GANHO_TREINO = {int(r.processos): float(r.ganho) for r in tab_par.itertuples()}
print(f"\n== Treinos simultâneos (cada treino com {N_SENS:,} linhas) =="); display(tab_par.round(3)); print(f"→ número recomendado de treinos simultâneos neste computador: {K_REC} (ganho {GANHO_TREINO[K_REC]:.1f}×)")

# ---------- 6.2 Previsão em vários processos ----------
pre_m, svm_m = MODELO.named_steps["escala"], MODELO.named_steps["svm"]; Xt = pre_m.transform(X_va)
t0 = time.perf_counter(); sc_seq = svm_m.decision_function(Xt); t_seq = time.perf_counter() - t0
linhas_prev = [{"processos": 1, "tempo_s": t_seq, "ganho": 1.0, "escores_iguais": True, "queda_ram_gib": 0.0}]; barra = Barra(max(len(procs) - 1, 1), f"Previsão em vários processos (modelo com {int(svm_m.n_support_.sum()):,} vetores de suporte)")
for k in procs:
    if k == 1: continue
    Parallel(n_jobs=k)(delayed(time.sleep)(0.01) for _ in range(k)); barra.avancar(0, f"{k} processos...")
    with MedidorSistema() as ms:
        t0 = time.perf_counter(); partes = Parallel(n_jobs=k)(delayed(svm_m.decision_function)(c) for c in np.array_split(Xt, k)); wall = time.perf_counter() - t0
    sc_par = np.concatenate(partes); linhas_prev.append({"processos": k, "tempo_s": wall, "ganho": t_seq / wall, "escores_iguais": bool(np.allclose(sc_par, sc_seq, rtol=1e-9, atol=1e-9)), "queda_ram_gib": ms.queda_gib})
    barra.avancar(1, f"{k} processos: {wall:.1f} s (ganho {t_seq / wall:.1f}×)")
barra.fechar(); tab_prev = pd.DataFrame(linhas_prev); tab_prev["eficiencia"] = tab_prev.ganho / tab_prev.processos
bons = tab_prev[tab_prev.eficiencia >= 0.5]; K_REC_PREV = int(bons.loc[bons.ganho.idxmax(), "processos"]) if (len(bons) and bons.ganho.max() >= 1.2) else 1; GANHO_PREV = {int(r.processos): float(r.ganho) for r in tab_prev.itertuples()}
print(f"\n== Previsão da validação ({N_VA:,} linhas) em vários processos =="); display(tab_prev.round(3)); print(f"→ número recomendado de processos para prever: {K_REC_PREV} (ganho {GANHO_PREV[K_REC_PREV]:.1f}×)")

  ⚠ 2 processos precisariam de ~1.6 GiB e só há 1.3 GiB livres: pulado
  ⚠ 4 processos precisariam de ~3.2 GiB e só há 1.3 GiB livres: pulado
  ⚠ 6 processos precisariam de ~4.7 GiB e só há 1.3 GiB livres: pulado
  ⚠ 8 processos precisariam de ~6.3 GiB e só há 1.3 GiB livres: pulado
Treinos simultâneos (40,000 linhas cada): concluído em 13s

== Treinos simultâneos (cada treino com 40,000 linhas) ==


,processos,tempo_total_s,tempo_medio_por_treino_s,n_sv_igual_em_todos,n_sv,ram_livre_inicial_gib,queda_ram_gib,treinos_por_min,ganho,eficiencia
0,1,13.128,12.475,True,1252,1.349,0.275,4.57,1.0,1.0


→ número recomendado de treinos simultâneos neste computador: 1 (ganho 1.0×)


Previsão em vários processos (modelo com 2,152 vetores de suporte): concluído em 1min 10s

== Previsão da validação (60,154 linhas) em vários processos ==


,processos,tempo_s,ganho,escores_iguais,queda_ram_gib,eficiencia
0,1,31.880,1.000,True,0.000,1.000
1,2,17.351,1.837,True,0.195,0.919
2,4,17.345,1.838,True,0.294,0.460
3,6,14.688,2.170,True,0.504,0.362
4,8,15.735,2.026,True,0.712,0.253


→ número recomendado de processos para prever: 2 (ganho 1.8×)


## 7. Estimativas e recomendação
Com as medidas (tempo de treino e vetores de suporte × tamanho, custo por par e o ganho do paralelismo), estima o tempo do que vem pela frente. **São extrapolações**: a coluna `extrapolado` marca as que usam tamanhos maiores que o maior medido. Valem como ordem de grandeza, e a primeira execução real da busca dirá o resto.
- **Busca com validação separada**: 9 treinos no ajuste (grade 3 × 3) + 9 previsões na validação.
- **Busca com CV de 5 dobras**: 45 treinos em 80% do ajuste + 45 previsões nos 20%.
- **Refit**: 1 treino com as 101.536 linhas do refit.
- **Previsão do teste**: 7,2 milhões de linhas × vetores de suporte do modelo final × custo por par, dividido pelo ganho da previsão em paralelo.
- **Multiclasse (referência)**: o SVM resolve 7 classes treinando **21 SVMs de pares** (cada par com 2 × 50 mil = 100 mil linhas no treino completo). O multiclasse ainda não existe: os números servem só para dimensionar o plano.
Os tempos "em paralelo" usam o número recomendado de processos (`K_REC`) e supõem que os treinos sejam independentes.

In [14]:
# ---------- 7.1 Estimativas ----------
proc_txt = lambda k: f"{k} processo" + ("" if k == 1 else "s")
S_T, S_P = GANHO_TREINO.get(K_REC, 1.0), GANHO_PREV.get(K_REC_PREV, 1.0)
prev_s = lambda linhas, n_sv: linhas * n_sv * NS_PAR * 1e-9                       # segundos para prever `linhas` com um modelo de `n_sv` vetores de suporte, 1 processo
PARES_MC = 21; PAR_CHEIO, PAR_DOBRA, PAR_AMOSTRA = 100_000, 80_000, int(round(2 * 100_000 / 7))
E = {}
def item(chave, nome, n_trein, tam, seq, par, maior_n, ganho_par):
    E[chave] = {"item": nome, "treinos": n_trein, "tamanho de cada": tam, "1_processo_s": float(seq), "paralelo_s": float(par), "extrapolado": bool(maior_n > N_MAX_MEDIDO * 1.05), "processos_paralelo": K_REC if ganho_par else 1}
item("busca_val", "Busca com validação separada (9 treinos)", 9, f"{N_AJ:,}", 9 * (f_t(N_AJ) + prev_s(N_VA, f_sv(N_AJ))), 9 * (f_t(N_AJ) + prev_s(N_VA, f_sv(N_AJ))) / S_T, N_AJ, True)
item("busca_cv", "Busca com CV de 5 dobras (45 treinos)", 45, f"{int(0.8 * N_AJ):,}", 45 * (f_t(0.8 * N_AJ) + prev_s(0.2 * N_AJ, f_sv(0.8 * N_AJ))), 45 * (f_t(0.8 * N_AJ) + prev_s(0.2 * N_AJ, f_sv(0.8 * N_AJ))) / S_T, 0.8 * N_AJ, True)
item("refit", "Refit (1 treino)", 1, f"{N_RF:,}", f_t(N_RF), f_t(N_RF), N_RF, False)
item("teste", f"Previsão do teste ({N_TESTE:,} linhas)", 1, f"{int(f_sv(N_RF)):,} vetores de suporte", prev_s(N_TESTE, f_sv(N_RF)), prev_s(N_TESTE, f_sv(N_RF)) / S_P, N_RF, False)
item("mc_fit", "Multiclasse: 1 treino completo (21 pares de 100 mil)", 1, f"{PARES_MC} × {PAR_CHEIO:,}", PARES_MC * f_t(PAR_CHEIO), PARES_MC * f_t(PAR_CHEIO) / S_T, PAR_CHEIO, True)
item("mc_cv", "Multiclasse: CV de 5 dobras (45 treinos de 21 pares de 80 mil)", 45, f"{PARES_MC} × {PAR_DOBRA:,}", 45 * PARES_MC * f_t(PAR_DOBRA), 45 * PARES_MC * f_t(PAR_DOBRA) / S_T, PAR_DOBRA, True)
item("mc_reduzido", "Multiclasse, plano reduzido: 9 treinos em ~100 mil linhas + 1 refit completo", 10, f"{PARES_MC} × {PAR_AMOSTRA:,} (e {PAR_CHEIO:,} no refit)", 9 * PARES_MC * f_t(PAR_AMOSTRA) + PARES_MC * f_t(PAR_CHEIO), 9 * PARES_MC * f_t(PAR_AMOSTRA) / S_T + PARES_MC * f_t(PAR_CHEIO) / S_T, PAR_CHEIO, True)
tab_est = pd.DataFrame(E.values()); tab_est["1 processo"] = tab_est["1_processo_s"].map(fmt_tempo); tab_est[f"{K_REC} em paralelo"] = tab_est["paralelo_s"].map(fmt_tempo)
print(f"== Estimativas (tempo de treino ~ n^{b_t:.2f}; maior tamanho medido: {N_MAX_MEDIDO:,} linhas; custo de previsão {NS_PAR:.0f} ns por par; ganho do treino {S_T:.1f}× e da previsão {S_P:.1f}× com {proc_txt(K_REC)}) ==")
display(tab_est[["item", "treinos", "tamanho de cada", "1 processo", f"{K_REC} em paralelo", "extrapolado"]])
h = lambda c: E[c]["paralelo_s"] / 3600
print("\n== Recomendação (limites: busca ≤ %.1f h, previsão do teste ≤ %.1f h) ==" % (LIMITE_BUSCA_H, LIMITE_TESTE_H))
if h("busca_cv") <= LIMITE_BUSCA_H: REC = "cv5"; print(f"✔ SVM binário: a CV de 5 dobras (45 treinos) cabe: ~{fmt_tempo(E['busca_cv']['paralelo_s'])} com {proc_txt(K_REC)}. Pode ser o mesmo protocolo do RF.")
elif h("busca_val") <= LIMITE_BUSCA_H: REC = "validacao"; print(f"⚠ SVM binário: a CV completa não cabe (~{fmt_tempo(E['busca_cv']['paralelo_s'])}). Use a validação separada (~{fmt_tempo(E['busca_val']['paralelo_s'])}) e a CV de 5 dobras só nas finalistas.")
else: REC = "amostra"; print(f"✘ SVM binário: nem a validação separada cabe (~{fmt_tempo(E['busca_val']['paralelo_s'])}). Faça a busca numa amostra menor e retreine o melhor nos dados completos.")
print(("✔" if h("teste") <= LIMITE_TESTE_H else "⚠") + f" Previsão do teste: ~{fmt_tempo(E['teste']['paralelo_s'])} com {proc_txt(K_REC_PREV)} (~{fmt_tempo(E['teste']['1_processo_s'])} com 1)" + ("" if h("teste") <= LIMITE_TESTE_H else ": passa do limite; considere prever em lotes com checkpoint, mais processos ou um modelo com menos vetores de suporte."))
print(("✔" if h("mc_cv") <= LIMITE_BUSCA_H else "⚠") + f" Multiclasse: CV de 5 dobras ~{fmt_tempo(E['mc_cv']['paralelo_s'])}; plano reduzido ~{fmt_tempo(E['mc_reduzido']['paralelo_s'])}" + ("" if h("mc_cv") <= LIMITE_BUSCA_H else " → para o multiclasse, o plano reduzido (busca em amostra e retreino do melhor) é o indicado."))
if any(v["extrapolado"] for v in E.values()): print("(⚠ itens marcados 'extrapolado' usam tamanhos maiores que o maior medido: confirme com a primeira execução real.)")

== Estimativas (tempo de treino ~ n^2.88; maior tamanho medido: 81,228 linhas; custo de previsão 202 ns por par; ganho do treino 1.0× e da previsão 1.8× com 1 processo) ==


,item,treinos,tamanho de cada,1 processo,1 em paralelo,extrapolado
0,Busca com validação separada (9 treinos),9,"81,228",25min 50s,25min 50s,False
1,Busca com CV de 5 dobras (45 treinos),45,"64,982",1h 02min,1h 02min,False
2,Refit (1 treino),1,"101,536",4min 40s,4min 40s,True
3,"Previsão do teste (7,219,189 linhas)",1,"2,391 vetores de suporte",58min 03s,31min 36s,True
4,Multiclasse: 1 treino completo (21 pares de 100 mil),1,"21 × 100,000",1h 33min,1h 33min,True
5,Multiclasse: CV de 5 dobras (45 treinos de 21 pares de 80 mil),45,"21 × 80,000",36h 56min,36h 56min,False
6,"Multiclasse, plano reduzido: 9 treinos em ~100 mil linhas + 1 refit completo",10,"21 × 28,571 (e 100,000 no refit)",1h 56min,1h 56min,True



== Recomendação (limites: busca ≤ 6.0 h, previsão do teste ≤ 1.5 h) ==
✔ SVM binário: a CV de 5 dobras (45 treinos) cabe: ~1h 02min com 1 processo. Pode ser o mesmo protocolo do RF.
✔ Previsão do teste: ~31min 36s com 2 processos (~58min 03s com 1)
⚠ Multiclasse: CV de 5 dobras ~36h 56min; plano reduzido ~1h 56min → para o multiclasse, o plano reduzido (busca em amostra e retreino do melhor) é o indicado.
(⚠ itens marcados 'extrapolado' usam tamanhos maiores que o maior medido: confirme com a primeira execução real.)


## 8. Gravação e conferências
Grava as tabelas, as estimativas e o `calibracao_svm.json` (com o hardware e as versões das bibliotecas, para a calibração ser reproduzível e comparável) e confere: amostras estratificadas, normalizador sem vazamento, previsão e treinos em paralelo que reproduzem o resultado sequencial, e estimativas válidas.

In [15]:
# ---------- 8.1 Gravação e conferências ----------
hw = {"so": platform.platform(), "python": sys.version.split()[0], "cpu": platform.processor(), "nucleos_logicos": os.cpu_count()}
if psutil: hw.update({"nucleos_fisicos": psutil.cpu_count(logical=False), "ram_total_gib": psutil.virtual_memory().total / 2 ** 30, "ram_livre_no_inicio_gib": float(tab_par["ram_livre_inicial_gib"].iloc[0])})
sw = {"scikit-learn": sklearn.__version__, "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__, "psutil": None if psutil is None else psutil.__version__}
tab_tam.to_csv(PASTA / "calibracao_tamanhos.csv", index=False); tab_sens.to_csv(PASTA / "sensibilidade_c_gamma.csv", index=False); tab_norm.to_csv(PASTA / "comparacao_normalizadores.csv", index=False)
tab_par.to_csv(PASTA / "paralelismo_treino.csv", index=False); tab_prev.to_csv(PASTA / "paralelismo_previsao.csv", index=False); tab_est.drop(columns=[c for c in tab_est.columns if c.endswith("processo") or c.endswith("paralelo")]).to_csv(PASTA / "estimativas.csv", index=False); diag.to_csv(PASTA / "diagnostico_caudas.csv", index=False)
salvar_json({"run_id": RUN_ID, "data": datetime.now().isoformat(timespec="seconds"), "hardware": hw, "software": sw, "normalizador": {"escolhido": "StandardScaler só nas colunas numéricas, ajustado no próprio treino; indicadores _ausente em 0/1", "n_numericas": len(NUM), "n_indicadores": len(IND)},
             "parametros": {"tamanhos": [n for n in TAMANHOS], "processos": PROCESSOS, "c_valores": C_VALORES, "gamma_fatores": GAMMA_FATORES, "cache_mb": CACHE_MB, "orcamento_fit_s": ORCAMENTO_FIT_S, "orcamento_sens_s": ORCAMENTO_SENS_S},
             "medidas": {"maior_tamanho_medido": N_MAX_MEDIDO, "expoente_tempo": b_t, "expoente_vetores_suporte": b_sv, "ns_por_par": NS_PAR, "expoentes_supostos": SUPOSTO, "amostra_sensibilidade": N_SENS, "gamma_scale": gamma0, "fator_tempo_na_grade": FATOR_GRADE,
                         "processos_recomendados_treino": K_REC, "ganho_treino": GANHO_TREINO, "processos_recomendados_previsao": K_REC_PREV, "ganho_previsao": GANHO_PREV},
             "estimativas": E, "recomendacao_protocolo_binario": REC, "limites_h": {"busca": LIMITE_BUSCA_H, "teste": LIMITE_TESTE_H},
             "observacao": "calibração de custo; não escolhe hiperparâmetros. ROC-AUC, recall e falso alarme da validação são informativos"}, PASTA / "calibracao_svm.json")
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
verifica("Amostras estratificadas: BENIGN = 50% em cada tamanho", all(abs(float((y_aj[amostra(None if r['n'] >= N_AJ else r['n'])] == 0).mean()) - 0.5) < 0.01 for r in res_tam), f"({len(res_tam)} tamanhos)")
_i = amostra(3000); _pp = novo_pipeline().fit(X_aj[_i], y_aj[_i]); _X3 = _pp.named_steps["escala"].transform(X_aj[_i]); _sd = _X3[:, NUM].std(0)
verifica("Normalizador sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos", np.allclose(_X3[:, NUM].mean(0), 0, atol=1e-9) and bool(((np.abs(_sd - 1) < 1e-6) | (_sd == 0)).all()) and ((not IND) or np.array_equal(_X3[:, IND], X_aj[_i][:, IND])))
verifica("Vetores de suporte ≤ linhas de treino em todas as medidas", bool((tab_tam.n_sv <= tab_tam.n).all() and (tab_sens.dropna(subset=["n_sv"]).n_sv <= tab_sens.dropna(subset=["n_sv"]).n).all()))
verifica("ROC-AUC da validação entre 0 e 1 em todas as medidas", bool(tab_tam.roc_auc_val.between(0, 1).all() and tab_sens.dropna(subset=["roc_auc_val"]).roc_auc_val.between(0, 1).all()))
verifica("Previsão em paralelo reproduz exatamente os escores da sequencial", bool(tab_prev.escores_iguais.all()))
verifica("Treinos simultâneos dão o mesmo modelo (mesmo número de vetores de suporte) em todos os processos", bool(tab_par.n_sv_igual_em_todos.all()))
verifica("Estimativas finitas e positivas", bool(np.isfinite(tab_est[["1_processo_s", "paralelo_s"]].to_numpy()).all() and (tab_est[["1_processo_s", "paralelo_s"]].to_numpy() > 0).all()))
verifica("Arquivos gravados", all((PASTA / f).exists() for f in ("calibracao_tamanhos.csv", "sensibilidade_c_gamma.csv", "comparacao_normalizadores.csv", "paralelismo_treino.csv", "paralelismo_previsao.csv", "estimativas.csv", "calibracao_svm.json", "diagnostico_caudas.csv")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA}")
print("\nPróximo passo: com estes números definimos o protocolo da busca do SVM (notebook 11).")

✔ Amostras estratificadas: BENIGN = 50% em cada tamanho (4 tamanhos)
✔ Normalizador sem vazamento: numéricas com média 0 e desvio 1 no treino; indicadores _ausente intactos 
✔ Vetores de suporte ≤ linhas de treino em todas as medidas 
✔ ROC-AUC da validação entre 0 e 1 em todas as medidas 
✔ Previsão em paralelo reproduz exatamente os escores da sequencial 
✔ Treinos simultâneos dão o mesmo modelo (mesmo número de vetores de suporte) em todos os processos 
✔ Estimativas finitas e positivas 
✔ Arquivos gravados 

✔ 8/8 checagens aprovadas. Resultados em C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\svm_binario_calibracao\svm_cal_20261024_224947

Próximo passo: com estes números definimos o protocolo da busca do SVM (notebook 11).
